# Этап 1. Аудит и подготовка корпуса

In [1]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import ftfy
import markdown
import pandas as pd
from bs4 import BeautifulSoup
from ftfy.badness import badness

In [2]:
data = json.loads(Path("filmoskop_kb.json").read_text(encoding="utf-8"))
docs_df = pd.DataFrame(data["docs"])
queries = data["queries"]

In [3]:
display(docs_df.count())
display(docs_df["format"].value_counts())
display(docs_df["category"].value_counts())
display(docs_df.assign(raw_len=docs_df["raw"].str.len()).groupby("format")["raw_len"].describe())

doc_id      181
title       181
category    181
format      181
raw         181
dtype: int64

format
text        132
markdown     28
html         21
Name: count, dtype: int64

category
Устройства и качество     42
Подписка и оплата         38
Аккаунт и профили         35
Функции и контент         34
Ошибки воспроизведения    32
Name: count, dtype: int64

,count,mean,std,min,25%,50%,75%,max
format,,,,,,,,
html,21.0,1014.523810,945.771147,42.0,399.00,570.0,1520.0,2818.0
markdown,28.0,980.035714,608.268184,420.0,506.25,802.5,1190.5,3057.0
text,132.0,733.356061,449.396170,14.0,421.25,659.0,1014.5,3017.0


### Метрики качества текста

In [4]:
MARKUP_PATTERNS = {
    "html": r"<[^>]+>|&\w+;",
    "markdown": r"(?m)^#+\s|\*\*|__|`",
    "text": r"<[^>]+>|&\w+;",  # в text тоже не должно быть остатков разметки
}

# слова, где кириллица смешана с латиницей: «пpoмокод», «подпискa»
MIXED_RE = re.compile(r"\w*(?:[а-яё]\w*[a-z]|[a-z]\w*[а-яё])\w*", re.IGNORECASE)
LOOKALIKE = str.maketrans("aceopxyABCEHKMOPTX", "асеорхуАВСЕНКМОРТХ")

DEFECTS = ["nbsp", "invisible", "mojibake", "replacement", "hyphen_wrap", "hyphen_dangling", "homoglyph"]

def markup_chars(text: str, fmt: str) -> int:
    """Число символов, занятых остатками разметки."""
    return sum(len(m.group()) for m in re.finditer(MARKUP_PATTERNS[fmt], text))

def invisible_chars(text: str) -> int:
    """Невидимые символы (Unicode-категория Cf: ZWSP, soft hyphen, BOM и т.д.)."""
    return sum(1 for ch in text if unicodedata.category(ch) == "Cf")

def audit(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Признаки качества по каждому документу."""
    s = df[col]
    out = pd.DataFrame({"doc_id": df["doc_id"], "format": df["format"]})
    out["len"] = s.str.len() # длина текста
    out["nbsp"] = s.str.count("\u00a0") # количество символов nbsp
    out["hyphen_wrap"] = s.str.count(r"\w-\n\w")  # перенос слова по слогам
    out["hyphen_dangling"] = s.str.count(r"[.!?…»]-[ \t]*\n")  # висячий дефис после абзаца
    out["homoglyph"] = s.map(lambda t: len(MIXED_RE.findall(t)))  # латиница внутри русских слов
    out["invisible"] = s.map(invisible_chars) # количество невидимых символов
    out["mojibake"] = s.map(badness) # количество кракозябр, mojibake
    out["replacement"] = s.str.count("\ufffd")  # потерянные символы
    out["markup"] = [markup_chars(t, f) for t, f in zip(s, df["format"])] # количество символов разметки
    out["index_dirty"] = out[DEFECTS].sum(axis=1) / out["len"] # общий индекс грязного текста
    out["index_markup"] = out["markup"] / out["len"] # индекс остатков разметки
    return out

def report(feats: pd.DataFrame) -> pd.DataFrame:
    """Сводка по формату: сколько документов с проблемой и средняя доля."""
    g = feats.groupby("format")
    return pd.DataFrame({
        "cnt_docs": g.size(),
        "cnt_dirty_docs": g["index_dirty"].apply(lambda s: (s > 0).sum()),
        "avg_index_dirty": g["index_dirty"].mean(),
        "markup_docs": g["markup"].apply(lambda s: (s > 0).sum()),
        "avg_index_markup": g["index_markup"].mean(),
        "total_len": g["len"].sum(),
    })

def defects_table(feats: pd.DataFrame) -> pd.DataFrame:
    """Сколько документов затронуто каждым дефектом, по формату."""
    return (feats[DEFECTS] > 0).groupby(feats["format"]).sum()


**Метрики текста до этапа очистки**

In [5]:
feats_before = audit(docs_df, "raw")
display(report(feats_before).style.set_caption("Сводная таблица дефектов в разрезе форматов"))
display(defects_table(feats_before).style.set_caption("Сколько документов затронуто каждым дефектом, по формату"))
display(feats_before.sort_values("index_dirty", ascending=False).head(10).style.set_caption("ТОП-10 документов по количеству дефектов"))

,cnt_docs,cnt_dirty_docs,avg_index_dirty,markup_docs,avg_index_markup,total_len
format,,,,,,
html,21,0,0.000000,21,0.226076,21305
markdown,28,4,0.029321,28,0.002782,27441
text,132,26,0.048416,0,0.000000,96803


,nbsp,invisible,mojibake,replacement,hyphen_wrap,hyphen_dangling,homoglyph
format,,,,,,,
html,0,0,0,0,0,0,0
markdown,1,0,2,0,1,1,0
text,5,5,15,1,3,0,4


,doc_id,format,len,nbsp,hyphen_wrap,hyphen_dangling,homoglyph,invisible,mojibake,replacement,markup,index_dirty,index_markup
43,43,text,860,0,0,0,0,0,388,0,0,0.451163,0.000000
124,124,text,661,0,0,0,0,0,297,0,0,0.449319,0.000000
46,46,text,604,0,0,0,0,0,270,0,0,0.447020,0.000000
111,111,text,777,0,0,0,0,0,347,0,0,0.446589,0.000000
38,38,text,847,0,0,0,0,0,378,0,0,0.446281,0.000000
54,54,text,683,0,0,0,0,0,304,0,0,0.445095,0.000000
112,112,text,881,1,0,0,0,0,390,0,0,0.443814,0.000000
22,22,text,2961,0,0,0,0,0,1314,0,0,0.443769,0.000000
106,106,text,717,0,0,0,0,1,316,0,0,0.442120,0.000000
178,178,text,1257,0,0,0,0,0,553,0,0,0.439936,0.000000


##### Доп. исследования перед очисткой

**Какие слова склеит исправление переносов** (смотрим глазами: нет ли настоящих дефисных слов)

In [6]:
joined = Counter(
    m.group(1) + m.group(2)
    for r in docs_df["raw"]
    for m in re.finditer(r"(\w+)-\n(\w+)", r)
)
print(len(joined), "уникальных слов")
print(sorted(joined))

23 уникальных слов
['Подписка', 'Фильмоскоп', 'автоматически', 'аккаунта', 'возвращения', 'интерфейса', 'максимальное', 'настройках', 'отключено', 'отличается', 'оформлении', 'переключиться', 'подключалась', 'подключение', 'подключения', 'помесячной', 'пребывания', 'рекомендации', 'сохранится', 'сохраняются', 'списывается', 'требуется', 'устройстве']


**Какие слова считаются латинскими двойниками**


In [7]:
mixed_words = Counter(w for r in docs_df["raw"] for w in MIXED_RE.findall(r))
print(len(mixed_words), "уникальных слов")
print(sorted(mixed_words))

29 уникальных слов
['aккaунтам', 'eщe', 'oплата', 'pаз', 'pедактировать', 'yкaзанную', 'Иcтория', 'Электрoнный', 'автоматичеcки', 'адpеса', 'вpучную', 'всeх', 'выгрузкa', 'мoнитoринга', 'мультипpофильным', 'нeделю', 'настрoек', 'оплaта', 'пpoмокод', 'плaтежа', 'плaтежей', 'подпискa', 'почтy', 'правильнocть', 'пришeл', 'разделe', 'спoсоб', 'хpанится', 'чeк']


**Анализ битых символов**

In [8]:
with pd.option_context("display.max_colwidth", None):
 display(docs_df[docs_df["raw"].str.count("\ufffd") > 0][["doc_id", "raw"]])

,doc_id,raw
120,120,"«Фильмоскоп» присылает два типа уведомлений. Сервисные — о списаниях, входах в аккаунт и изменениях подписки — отключить н�льзя: они защища�т акк�унт и деньги. Рекламные — о новинках, подборках и акциях — настраиваются.\n\nУправление в настройк�х профиля, раздел «Уведомлени�»: отдельно включаются и отключаются письма на почту и пуш-уведомления на устройствах. Уведомления о выходе новых серий настраиваются в карточке сериала кнопкой «Н�помнить»."


С такой проблемой всего 1 документ, поэтому можно исправить с помощью простой замены по словарю

In [9]:
MANUAL_FIXES = {
    "н\ufffdльзя": "нельзя",
    "защища\ufffdт": "защищают",
    "акк\ufffdунт": "аккаунт",
    "настройк\ufffdх": "настройках",
    "Уведомлени\ufffd": "Уведомления",
    "Н\ufffdпомнить": "Напомнить",
}

### Функции очистки текста

In [10]:
def clean_html(text):
    """Убираем теги футера, навигации, боковых панелей, скриптов и стилей
    Для текущего набора данных: Проанализировав содержимое текта определил что полезный текст находится или в теге article
    или в теге div с классом kb-article, поэтому в первую очередь пробуем взять его
    """
    soup = BeautifulSoup(text, "html.parser")
    for tag in soup(["footer", "head", "nav", "aside", "script", "style"]):
        tag.decompose()
    root = soup.select_one("article, div.kb-article") or soup
    return root.get_text()

def clean_markdown(text: str) -> str:
    """markdown -> html -> текст: синтаксис убирает библиотека."""
    soup = BeautifulSoup(markdown.markdown(text), "html.parser")
    for lst in soup.find_all(["ol", "ul"]):
        for i, li in enumerate(lst.find_all("li", recursive=False), 1):
            li.insert(0, f"{i}. " if lst.name == "ol" else "- ")
    return soup.get_text()

def clean_text(text: str) -> str:
    """Plain text: специфичной разметки нет, чистит общая часть."""
    return text

CLEANERS = {"html": clean_html, "markdown": clean_markdown, "text": clean_text}

def fix_chars(text: str) -> str:
    """Mojibake (ftfy), NBSP -> пробел, удаление невидимых символов"""
    text = ftfy.fix_text(text, uncurl_quotes=False)
    for bad, good in MANUAL_FIXES.items():
        text = text.replace(bad, good)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text) # удаляем переносы слов
    text = re.sub(r"(?<=[.!?…»])-(?=[ \t]*\n)", "", text)  # удаляем висячий дефис после абзаца
    return "".join(ch for ch in text if unicodedata.category(ch) != "Cf")

def fix_homoglyphs(text: str) -> str:
    """Латинские двойники -> кириллица, только внутри слов со смешанными алфавитами."""
    return MIXED_RE.sub(lambda m: m.group().translate(LOOKALIKE), text)

def normalize_ws(text: str) -> str:
    """Схлопывает пробелы, чистит края строк, оставляет \n\n между абзацами."""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[ \t]+", " ", text)                          # лишние пробелы внутри строки
    text = "\n".join(line.strip() for line in text.splitlines())  # пробелы по краям строк
    return re.sub(r"\n{3,}", "\n\n", text).strip()               # много пустых строк -> одна


def clean(raw: str, fmt: str) -> str:
    """Порядок: кодировка -> разметка по формату -> двойники -> пробелы."""
    return normalize_ws(fix_homoglyphs(CLEANERS[fmt](fix_chars(raw))))



#### Этап очистки

In [11]:
docs_df["text"] = [clean(r, f) for r, f in zip(docs_df["raw"], docs_df["format"])]

**Метрики текста после очистки**

In [12]:
feats_after = audit(docs_df, "text")
display(pd.concat({"before": report(feats_before), "after": report(feats_after)}, axis=1))
display(pd.concat({"before": defects_table(feats_before), "after": defects_table(feats_after)}, axis=1))

before                                                              \
         cnt_docs cnt_dirty_docs avg_index_dirty markup_docs avg_index_markup   
format                                                                          
html           21              0        0.000000          21         0.226076   
markdown       28              4        0.029321          28         0.002782   
text          132             26        0.048416           0         0.000000   

                      after                                             \
         total_len cnt_docs cnt_dirty_docs avg_index_dirty markup_docs   
format                                                                   
html         21305       21              0             0.0           0   
markdown     27441       28              0             0.0           0   
text         96803      132              0             0.0           0   

                                     
         avg_index_markup total_len  
format                               
html                  0.0     13916  
markdown              0.0     25476  
text                  0.0     89646

before                                                             \
           nbsp invisible mojibake replacement hyphen_wrap hyphen_dangling   
format                                                                       
html          0         0        0           0           0               0   
markdown      1         0        2           0           1               1   
text          5         5       15           1           3               0   

                   after                                             \
         homoglyph  nbsp invisible mojibake replacement hyphen_wrap   
format                                                                
html             0     0         0        0           0           0   
markdown         0     0         0        0           0           0   
text             4     0         0        0           0           0   

                                    
         hyphen_dangling homoglyph  
format                              
html                   0         0  
markdown               0         0  
text                   0         0

**Доп проверки качества текста после очистки**

In [13]:
# html, у которых не нашёлся article: остались без вырезания контента
no_article = [
    d for d, r, f in zip(docs_df["doc_id"], docs_df["raw"], docs_df["format"])
    if f == "html" and not BeautifulSoup(r, "html.parser").find(["article"])
    and not BeautifulSoup(r, "html.parser").find("div", class_="kb-article")
] or 0
print("html без article/kb-article:", no_article)

html без article/kb-article: 0


In [14]:
# посчитаем количество цифр в документах до и после очистки, чтоб убедиться что не потеряли артикулы, цены и подобные значения
# исключим только числа которые могли быть внутри тегов html по типу <h1> <a href='site123.com...

def numbers(text: str) -> set[str]:
    matches = re.finditer(r"\d+", text)
    result = {
        match.group()
        for match in matches
        if not re.search(r"<[^>]*$", text[:match.start()])
    }
    return result

lost = {
    d: sorted(numbers(r) - numbers(t))
    for d, r, t in zip(docs_df["doc_id"], docs_df["raw"], docs_df["text"])
    if numbers(r) - numbers(t)
}
print("Документов с потерянными числами:", len(lost))

Документов с потерянными числами: 0


In [15]:
# проверим документы на дубликаты
print(docs_df["text"].is_unique)

True


In [16]:
# распределение длин очищеного текста, пустые строки
display(docs_df["text"].str.len().describe())

count     181.000000
mean      712.917127
std       421.620154
min        14.000000
25%       416.000000
50%       570.000000
75%       999.000000
max      1851.000000
Name: text, dtype: float64

In [17]:
# визуально посмотрим топ 5 самых коротких документов
with pd.option_context("display.max_colwidth", None):
    display(
        docs_df[["doc_id", "format", "text", "raw"]]
        .assign(text_length=docs_df["text"].str.len())
        .sort_values("text_length")
        .head(15)
    )

,doc_id,format,text,raw,text_length
153,153,text,QX-500;2020;да,QX-500;2020;да,14
156,156,html,Схема разделов,"<div class=""kb-article""><h1>Схема разделов",14
145,145,text,"Если ошибка мешает просмотру, обновите приложение до последней версии.","Если ошибка мешает просмотру, обновите приложение до последней версии.",70
71,71,html,Статья обновляется\nРаздел временно недоступен Статья обновляется. Загляните позже.,"<div class=""kb-article""><h1>Статья обновляется</h1>\n<p>Раздел временно недоступен Статья обновляется. Загляните позже.</p>\n</div>",82
74,74,html,Раздел в разработке\nРаздел в разработке Мы готовим этот раздел. Информация появится в ближайшее время.,"<div class=""kb-article""><h1>Раздел в разработке</h1>\n<p>Раздел в разработке Мы готовим этот раздел. Информация появится в ближайшее время.</p>\n</div>",102
53,53,text,"Раздел перенесён.\n\nСм. приложение 2 к регламенту размещения справочных материалов.\n\n[черновик, не публиковать]","Раздел перенесён.\n\nСм. приложение 2 к регламенту размещения справочных материалов.\n\n[черновик, не публиковать]",110
52,52,html,"Раздел справки (архив)\nСтраница не найдена Возможно, раздел был перемещён или удалён. Воспользуйтесь поиском по справке.","<div class=""kb-article""><h1>Раздел справки (архив)</h1>\n<p>Страница не найдена Возможно, раздел был перемещён или удалён. Воспользуйтесь поиском по справке.</p>\n</div>",120
39,39,text,"ticket_id;category;status;resolved_hours 10234;оплата;closed;2 10235;оплата;closed;5 10241;подписка;open; 10250;возврат;closed;26 10252;оплата;closed;1 10260;промокод;closed;3\n\nЭкспорт обращений за неделю, выгружено автоматически.","ticket_id;category;status;resolved_hours 10234;оплата;closed;2 10235;oплата;closed;5 10241;подпискa;open; 10250;возврат;closed;26 10252;оплaта;closed;1 10260;пpoмокод;closed;3\n\nЭкспорт обращений за нeделю, выгружено автоматически.",230
75,75,text,"{""service"": ""player-api"", ""version"": ""2.14.3"", ""env"": ""production"", ""features"": {""subtitles_v2"": true, ""preload"": false}, ""updated_at"": ""2026-02-11T09:14:00Z""}\n\nКонфигурация выгружена системой мониторинга. Не редактировать вручную.","{""service"": ""player-api"", ""version"": ""2.14.3"", ""env"": ""production"", ""features"": {""subtitles_v2"": true, ""preload"": false}, ""updated_at"": ""2026-02-11T09:14:00Z""}\n\nКонфигурация выгружена системой мoнитoринга. Не pедактировать вpучную.",231
127,127,text,Помогите нам стать лучше — пройдите короткий опрос о работе личного кабинета.\n\nОпрос занимает три минуты и открывается по ссылке из письма или из баннера в настройках аккаунта. Ответы анонимны и используются только для улучшения сервиса.,Помогите нам стать лучше — пройдите короткий опрос о работе личного кабинета.\n\nОпрос занимает три минуты и открывается по ссылке из письма или из баннера в настройках аккаунта. Ответы анонимны и используются только для улучшения сервиса.,237


Судя по отчёту в корпусе есть мусорные документы, обрывки текста, архивные статьи, какие то служебные заметки для внутреннего пользования. Видно что и до очистки они были такими же. Этап очистки тут лишнего не удалил.
На данном этапе оставлю эти данные как есть.

### «Было → стало» на конкретных разделах

In [18]:
def show_diff(doc_id: int, width: int = 450) -> None:
    row = docs_df.set_index("doc_id").loc[doc_id]
    print(f"doc {doc_id} [{row['format']}] {row['title']}")
    print("RAW  :", repr(row["raw"][:width]))
    print("CLEAN:", repr(row["text"][:width]), "\n")


examples = (
    docs_df.loc[feats_before["hyphen_wrap"] > 0, "doc_id"].head(1).tolist()
    + docs_df.loc[feats_before["replacement"] > 0, "doc_id"].head(1).tolist()
    + docs_df.loc[feats_before["mojibake"] > 0, "doc_id"].head(1).tolist()
    + docs_df.query("format == 'html'")["doc_id"].head(1).tolist()
    + docs_df.query("format == 'markdown'")["doc_id"].head(1).tolist()
)
for d in examples:
    show_diff(d)

doc 20 [text] Просмотр за границей
RAW  : 'Готовить что-то перед поездкой не\xa0нужно: аккаунт,\xa0профили и\xa0привязанные устройства остаются прежними,\xa0а списания за подписку проходят в обычные сроки с той же карты. Предупреждать сервис об отъезде тоже не требуетс-\nя.\n\nЯзык интер-\nфейса, субтитры и остальные настройки профиля от страны не зависят —\xa0они остаются такими,\xa0какими вы выбрали их дома. Список\xa0«Моё» и оценки фильмов\xa0в поездке тоже сохран-\nяются. Переключать\xa0что-либо в настройках после\xa0п'
CLEAN: 'Готовить что-то перед поездкой не нужно: аккаунт, профили и привязанные устройства остаются прежними, а списания за подписку проходят в обычные сроки с той же карты. Предупреждать сервис об отъезде тоже не требуется.\n\nЯзык интерфейса, субтитры и остальные настройки профиля от страны не зависят — они остаются такими, какими вы выбрали их дома. Список «Моё» и оценки фильмов в поездке тоже сохраняются. Переключать что-либо в настройках после приезда' 

doc 12

### Итоги:
1. В результате анализа документов были определены проблемы и способы их решения
Остатки разметки документов
Наличие в тексте невидимых символов по типу `non-break space` `zero width space` кракозябры они же mojibake переносы строк 
определены следующие метрики качества в разрезе каждого типа данных процент остатков разметки, и процент грязных символов(невидимые пробелы, кракозябры, непечатные символы, переносы строк). 
разработана функция очистки
сначала идет процесс не зависимо от типа документа это восстановления символов, восстановление кодировки удаление пробелов
потом в зависимости от формата документа вызывается функция удаления разметки потом функция нормализации текста, удаление лишних пробелов, преобразование nfc unicode
для случая когда невозможно восстановить текст делаем вручную, для1 документа
после очистки проводятся доп проверки на html, у которых не нашёлся article: остались без вырезания контента, # проверим документы на дубликаты, смотрим на отчет # распределение длин очищеного текста, пустые строки, глазами смотрим на самые короткие, удостоверямся что короткий текст не по причине очистки
и в финале смотрим ещё раз сравнение отчёта по метрика до и после и видим что по каждому формату докумнетов оба значения cnt_dirty_docs документы с грязными символами- 	markup_docs документы со следами разметки,	равны 0
так же для наглядности можно пробежать ся глазами по итоговому результату очистки
в конце я сохраню очищенный набор в качестве елевого и на след этапах буду работать с ним


### Сохранение очищенного корпуса

In [19]:
docs_df[["doc_id", "title", "category", "format", "text"]].to_json(
    "docs_clean.jsonl", orient="records", lines=True, force_ascii=False, indent=2
)
print("Сохранено:", len(docs_df), "документов")

Сохранено: 181 документов


# Отчёт

## Данные

В корпусе 181 раздел базы знаний, 5 категорий и три формата выгрузки: text (132), markdown (28), html (21). Из-за разных источников выгрузки дефекты в форматах разные, поэтому метрики и очистку я строил по формату.

## Найденные проблемы и метрики

| Проблема | Формат | Как обнаружил |
|---|---|---|
| Невидимые символы: NBSP, ZWSP и другие (категория Unicode `Cf`) | text | подсчёт символов |
| Mojibake (испорченная кодировка) | text, markdown | `ftfy.badness`, просмотр примеров |
| Потерянные символы (`U+FFFD`) | text, 1 документ (doc 120) | подсчёт `U+FFFD` |
| Переносы слов по слогам (`оформ-\nлении`) и висячие дефисы после абзаца | markdown (например, doc 162) | паттерны `\w-\n\w` и `[.!?…»]-\s*\n` |
| Латинские двойники внутри русских слов (`пpoмокод`, `подпискa`) | text (например, docs 39, 75, 80) | паттерн слов со смешанными алфавитами |
| Остатки разметки: теги, сущности, markdown-синтаксис | html, markdown | подсчёт символов разметки по паттернам формата |
| Обёртка html-страницы: `nav`, `footer`, `aside`, `script`, `style` | html | ручной просмотр |


Для каждого документа считаю два индекса:

- **Индекс грязного текста** = (NBSP + невидимые символы + mojibake + `U+FFFD` + переносы слов + висячие дефисы + латинские двойники) / длина текста
- **Индекс остатков разметки** = символы, занятые разметкой, / длина текста. Паттерны зависят от формата.

Документ считаю проблемным, если метрика больше нуля: любое вхождение перечисленных дефектов само по себе дефект, поэтому порог не нужен. Среднее по всему корпусу скрывает локализацию дефектов, поэтому я считаю метрики по документам и агрегирую по форматам: так видно и долю затронутых документов, и худшие из них.

## Принятые решения

Порядок очистки: кодировка → разметка по формату → латинские двойники → пробелы. Иначе парсеры и регулярки работают по уже искажённому тексту. Принцип: исправляю причину дефекта, документы не удаляю.

1. **Кодировка:** `ftfy.fix_text` исправляет mojibake обратной перекодировкой, а не вырезает его.
2. **Невидимые символы:** NBSP заменяю обычным пробелом (иначе слова склеились бы), остальные символы категории `Cf` удаляю.
3. **Переносы:** склеиваю слова, разорванные по слогам, висячие дефисы удаляю. Список склеенных слов просмотрел вручную: настоящих дефисных слов среди них нет.
4. **Латинские двойники:** заменяю на кириллицу только внутри слов со смешанными алфавитами, чисто латинские токены (`player-api`, `ticket_id`) не трогаю.
5. **Разметка по формату:**
   - html: отбрасываю обёртку страницы и беру содержимое `article` или `div.kb-article`; расположение полезного текста определил ручным просмотром;
   - markdown: `markdown` → html → текст, номера и маркеры списков сохраняю, чтобы не терять порядок шагов в инструкциях;
   - text: специфичной разметки нет, достаточно общей части.
6. **Пробелы:** NFC, схлопывание пробелов внутри строки, одна пустая строка между абзацами. Границы абзацев сохраняю для чанкинга.
7. **Потерянные символы (doc 120):** буквы утрачены при выгрузке, автоматически их не восстановить, а окончание зависит от смысла предложения. Восстановил вручную через словарь замен слов, который применяется внутри общей функции очистки, поэтому результат воспроизводим.

## Результат: было → стало

| Формат | Проблемных документов | Индекс грязного текста | Документов с остатками разметки | Индекс остатков разметки |
|---|---|---|---|---|
| html | 0 → 0 | 0 → 0 | 21 → 0 | 0,226 → 0 |
| markdown | 4 → 0 | 0,029 → 0 | 28 → 0 | 0,0028 → 0 |
| text | 22 → 0 | 0,048 → 0 | 0 → 0 | 0 → 0 |


Что подтверждает успех очистки:

- оба индекса равны нулю по всем форматам;
- проверка, что очистка не потеряла числа, цены и коды: потерянных чисел 0;
- html без `article` / `kb-article`: 0;
- результат на конкретных разделах просмотрен вручную;

## Вывод

Я провёл аудит корпуса из 181 раздела и нашёл семь типов дефектов: невидимые символы, mojibake, потерянные символы, переносы слов, латинские двойники, остатки разметки и обёртку html. Собрал пайплайн из общей части (кодировка, символы, переносы, двойники) и части по формату (html, markdown, text). После очистки метрики загрязнённости и остатков разметки по всем форматам равны нулю, абзацная структура сохранена для следующего этапа. Дефекты исправлял по причине (перекодировка, склейка слов, замена двойников), а не вырезанием симптомов, и ничего не удалял. Единственный случай, где данные нельзя восстановить автоматически (doc 120), исправлен вручную и задокументирован. Очищенный корпус сохранён в `docs_clean.jsonl`, дальнейшие этапы работают с ним.